# 🧪 Лабораторна робота №3
## Тема: Таймери, апаратні переривання та послідовний зв'язок (UART)

## 🎯 Мета роботи:
- Зрозуміти проблему блокуючого коду (`time.sleep`) і навчитися її уникати
- Навчитися працювати з апаратними перериваннями через `Pin.irq()`
- Навчитися виконувати періодичні задачі через `machine.Timer`
- Організувати обмін даними по UART — між ESP32 і комп'ютером, а також між двома платами

---

## 📖 Розширені теоретичні відомості

### 1️⃣ Проблема блокуючого коду
- У Лабораторній №1 ми використовували `time.sleep()` — але поки він триває, плата **нічого іншого не робить**: не реагує на кнопки, не оновлює дисплей, не приймає дані.
- Наприклад, якщо один світлодіод блимає з `sleep(1)`, а нам одночасно треба перевіряти кнопку — програма буде «пропускати» швидкі натискання, поки чекає.
- Розв'язання: **переривання** (реагують миттєво на подію) та **таймери** (виконують задачу періодично «у фоні», не блокуючи основний цикл).

### 2️⃣ Апаратні переривання: `Pin.irq()`
```python
from machine import Pin

def on_press(pin):
    print("Переривання! Пін:", pin)

button = Pin(15, Pin.IN, Pin.PULL_UP)
button.irq(trigger=Pin.IRQ_FALLING, handler=on_press)
```

- `trigger` визначає подію: `Pin.IRQ_RISING` (перепад 0→1), `Pin.IRQ_FALLING` (перепад 1→0), або обидва разом (`Pin.IRQ_RISING | Pin.IRQ_FALLING`).
- `handler` — функція, яка викликається **автоматично**, щойно відбудеться подія, незалежно від того, що зараз виконує основний код.
- **Важливо:** обробник переривання (ISR) має бути **дуже коротким** — не використовуйте в ньому `time.sleep()`, складні обчислення чи виділення пам'яті. Найкраща практика — просто виставити прапорець (`flag = True`), а основну роботу виконати в головному циклі.

### 3️⃣ Антидребезг у перериваннях
- Оскільки переривання реагує миттєво, дребезг контактів кнопки викликає **кілька** переривань на одне натискання.
- Типовий підхід — перевіряти час від останнього спрацювання через `time.ticks_ms()` / `time.ticks_diff()`:
  ```python
  last_irq = 0

  def on_press(pin):
      global last_irq
      now = time.ticks_ms()
      if time.ticks_diff(now, last_irq) > 200:   # ігноруємо повтори швидше 200 мс
          print("Натискання зараховано")
          last_irq = now
  ```

### 4️⃣ `machine.Timer` — періодичні задачі
```python
from machine import Timer

led_state = False

def blink(timer):
    global led_state
    led_state = not led_state
    led.value(led_state)

tim = Timer(0)
tim.init(period=500, mode=Timer.PERIODIC, callback=blink)  # кожні 500 мс
```

- `period` — інтервал у мілісекундах; `mode` — `Timer.PERIODIC` (повторюється) або `Timer.ONE_SHOT` (одноразово).
- Таймер працює у фоні — основний `while True` цикл продовжує виконувати іншу логіку (наприклад, опитувати кнопки чи UART) без затримок.

### 5️⃣ Послідовний зв'язок UART
- **UART (Universal Asynchronous Receiver-Transmitter)** — простий протокол передачі даних побайтово через два дроти: `TX` (передача) і `RX` (прийом).
- Обидва пристрої повинні бути налаштовані на однакову **швидкість (baud rate)**, найпоширеніша — `115200`.
- Той самий UART, через який плата спілкується з комп'ютером у REPL/Serial Monitor (зазвичай `UART0`, пов'язаний з USB), можна використати і в коді — але для комунікації з **іншим** пристроєм (наприклад другою платою) варто взяти окремий, вільний `UART1`/`UART2` на довільних GPIO.
- **Framing:** дані зазвичай передають як текстові рядки з символом кінця рядка (`\n`), щоб приймач знав, де рядок закінчується.

### 6️⃣ Клас `machine.UART`
```python
from machine import UART

# UART2 на довільних пінах (TX=17, RX=16) — не заважає REPL на USB
uart = UART(2, baudrate=115200, tx=17, rx=16)

uart.write("Привіт від ESP32\n")

if uart.any():                 # чи є вхідні дані
    data = uart.readline()
    print("Отримано:", data)
```

- `uart.any()` повертає кількість байтів, доступних для читання (0, якщо нічого немає) — використовуйте це для неблокуючого читання.
- `uart.write()` приймає рядок або `bytes`.

### 7️⃣ Типові помилки початківців
- Виконують важку логіку (обчислення, `sleep`, друк багатьох рядків) прямо в обробнику переривання → плата «зависає» або видає `IRQ` помилки.
- Забувають антидребезг у перериваннях → одне натискання рахується кілька разів.
- Використовують той самий `UART0`, що й REPL, для власного протоколу → конфлікт із виведенням службових повідомлень.
- Забувають однакову швидкість (`baudrate`) на обох сторонах з'єднання → отримують «сміття» замість даних.
- Плутають `tx`/`rx` місцями при з'єднанні двох плат: `TX` однієї плати повинен йти до `RX` іншої, і навпаки (а не TX-TX).

### 8️⃣ Як перевіряти код цієї лабораторної роботи

У MicroPython немає локального "пісочного" середовища, як для звичайного Python — код виконується **на самому мікроконтролері** ESP32. Є два способи перевірити свої рішення.

**Варіант A. Jupyter Notebook + реальна плата ESP32 (рекомендовано, якщо є плата)**

1. Прошийте плату прошивкою MicroPython ([https://micropython.org/download/ESP32_GENERIC/](https://micropython.org/download/ESP32_GENERIC/)), один раз через `esptool.py`.
2. Встановіть ядро для Jupyter, яке вміє працювати з платою по USB/Serial:
   ```bash
   pip install jupyter_micropython_kernel
   python -m jupyter_micropython_kernel.install
   ```

3. Відкрийте цей ноутбук і оберіть ядро **MicroPython** (або **MicroPython - USB**) замість Python 3.
4. У першій комірці виконайте підключення до плати (замініть порт на свій — `COMx` у Windows, `/dev/ttyUSB0` або `/dev/ttyACM0` у Linux/macOS):
   ```
   %serialconnect --port=COM5 --baud=115200
   ```

5. Далі виконуйте комірки з кодом як завжди — код одразу запускається на платі, а `print()` повертає результат прямо в ноутбук. Світлодіоди й кнопки реагують фізично.
6. Якщо потрібно скинути з'єднання: `%disconnect`, а щоб побачити файли на платі: `%lsmagic`.

**Варіант B. Немає плати під рукою — симулятор Wokwi**

1. Відкрийте [https://wokwi.com](https://wokwi.com) → **New Project → ESP32 → MicroPython**.
2. У файлі `main.py` вставте код із завдання.
3. На вкладці **diagram.json** опишіть схему (перелік деталей і з'єднань наведено в кожному завданні нижче) — або зберіть її мишкою в графічному редакторі Wokwi, перетягуючи компоненти з панелі **Parts**.
4. Натисніть зелену кнопку **▶ Play** — почнеться симуляція. Вивід `print()` з'являється у вкладці **Serial Monitor**.
5. Перевірте результат за критеріями, вказаними у кожному завданні (що саме має блимати/показуватись/виводитись).
6. Готовий проєкт можна зберегти (**Save**) і надіслати викладачу посилання на нього — це і буде звіт з лабораторної.

**Схема для цієї лабораторної (Wokwi):** кнопка (Pushbutton) на `GPIO15` (для завдань з перериваннями), світлодіод з резистором на `GPIO2` (для таймера). Для завдань з UART: у Wokwi найпростіше перевірити обмін «плата ↔ комп'ютер» через вбудований **Serial Monitor** (він і є UART0/USB) — окремого дроту прокладати не потрібно. Для завдання «дві плати по UART» додайте другий ESP32 у той самий проєкт diagram.json і з'єднайте `GPIO17 (TX)` першої плати з `GPIO16 (RX)` другої, і навпаки `GPIO16 (RX)` першої з `GPIO17 (TX)` другої, а також спільний `GND`.

> 💡 Порада: пишіть і перевіряйте код невеликими фрагментами — спочатку переконайтесь, що плата взагалі відповідає (наприклад, `print("hello")`), і лише потім додавайте логіку.

---

## 📝 Завдання 1. Лічильник натискань через переривання
Реалізуйте підрахунок натискань кнопки (`GPIO15`) виключно через `Pin.irq()` — без опитування (`polling`) у циклі. Не забудьте про антидребезг за часом.

In [ ]:
from machine import Pin
import time

count = 0
last_irq = 0

def on_press(pin):
    global count, last_irq
    now = time.ticks_ms()
    if time.ticks_diff(now, last_irq) > 200:
        count += 1
        print("Натискань:", count)
        last_irq = now

button = Pin(15, Pin.IN, Pin.PULL_UP)
button.irq(trigger=Pin.IRQ_FALLING, handler=on_press)

while True:
    # основний цикл вільний для іншої роботи
    time.sleep(1)
    print("...програма працює далі, лічильник:", count)


### 🔢 Завдання:
1. Змініть поріг антидребезгу з 200 мс на 50 мс і поясніть у коментарі, як це впливає на результат.
2. Додайте другу кнопку з власним незалежним лічильником і власним обробником.
3. Реалізуйте реакцію на `Pin.IRQ_RISING` (відпускання) замість `IRQ_FALLING`.
4. У обробнику лише виставляйте прапорець (`flag = True`), а весь друк і логіку виконуйте в основному циклі — поясніть, чому так правильніше.
5. Додайте світлодіод, що вмикається на 100 мс при кожному зафіксованому натисканні (керування — з основного циклу, не з обробника).
6. Реалізуйте розпізнавання одинарного й подвійного клацання (за інтервалом між натисканнями).
7. Виміряйте й виведіть час між двома останніми натисканнями.
8. Зробіть так, щоб після 5 натискань обробник переривання від'єднувався (`irq(handler=None)`) і кнопка переставала реагувати.
9. Порівняйте (експериментально і в коментарі) поведінку з antibounce і без нього при швидких натисканнях.
10. Додайте обробку двох різних пінів одним і тим самим обробником, розрізняючи їх за `pin` (аргумент функції).

## 📝 Завдання 2. Неблокуюче блимання через Timer
Реалізуйте блимання світлодіодом (`GPIO2`) кожні 500 мс через `machine.Timer`, при цьому основний цикл одночасно повинен друкувати повідомлення раз на секунду — обидва процеси йдуть незалежно.

In [ ]:
from machine import Pin, Timer
import time

led = Pin(2, Pin.OUT)

def toggle_led(timer):
    led.toggle()

tim = Timer(0)
tim.init(period=500, mode=Timer.PERIODIC, callback=toggle_led)

counter = 0
while True:
    counter += 1
    print("Головний цикл, тік:", counter)
    time.sleep(1)


### 🔢 Завдання:
1. Змініть інтервал блимання на 100 мс і подивіться, чи головний цикл і далі друкує рівно раз на секунду.
2. Додайте другий таймер для другого світлодіода з іншим інтервалом.
3. Реалізуйте `Timer.ONE_SHOT`, що вимикає світлодіод рівно через 3 секунди після старту програми (одноразово).
4. Додайте кнопку, яка змінює період таймера (наприклад, перемикає між 200/500/1000 мс).
5. Реалізуйте лічильник спрацювань таймера і виводьте його в основному циклі.
6. Об'єднайте цю задачу з попереднім завданням: додайте кнопку з перериванням, яка працює одночасно з таймером.
7. Зупиніть таймер через `tim.deinit()` після 20 морганнь і виведіть повідомлення про завершення.
8. Реалізуйте на таймері «SOS» азбукою Морзе (виклик функції, яка кожного разу видає наступний символ послідовності).
9. Порівняйте точність блимання через `Timer` і через `time.sleep()` у циклі (в коментарі поясніть різницю).
10. Реалізуйте два незалежні світлодіоди, що блимають з відношенням частот 1:3, обидва — тільки через таймери.

## 📝 Завдання 3. Надсилання даних на ПК через UART
Реалізуйте псевдо-сенсор (наприклад, `random`-значення «температури») і надсилайте його показання через UART (Serial) раз на секунду у форматі `TEMP:23.4`.

In [ ]:
from machine import UART
import time
import urandom

uart = UART(0, baudrate=115200)   # UART0 = той самий канал, що й Serial Monitor / REPL

def fake_temperature():
    return 20 + urandom.getrandbits(8) / 255 * 10   # псевдо-значення 20.0-30.0

while True:
    t = fake_temperature()
    line = "TEMP:{:.1f}\n".format(t)
    uart.write(line)
    time.sleep(1)


### 🔢 Завдання:
1. Додайте другий псевдо-датчик (наприклад, вологість) і надсилайте обидва значення в одному рядку через кому.
2. Змініть формат повідомлення на простіший для розбору JSON-подібний рядок: `{"temp": 23.4}`.
3. Додайте до кожного повідомлення порядковий номер вимірювання.
4. Замініть псевдо-датчик на реальний потенціометр (ADC) — надсилайте справжнє зчитане значення.
5. Реалізуйте надсилання «аварійного» повідомлення, якщо значення виходить за межі 15-35.
6. Зменшіть інтервал надсилання до 200 мс і оцініть, чи встигає Serial Monitor / ноутбук усе показати без втрат.
7. Додайте до кожного повідомлення часову мітку (`time.ticks_ms()`).
8. Реалізуйте контрольну суму (наприклад, суму символів за модулем 256) в кінці рядка для перевірки цілісності.
9. Виводьте показання лише тоді, коли вони змінились більш ніж на 0.5 градуса відносно попереднього.
10. Задокументуйте у коментарі власний простий текстовий протокол (формат рядка) для цього обміну.

## 📝 Завдання 4. Прийом команд з ПК і керування LED
Реалізуйте прийом текстових команд через UART від комп'ютера (з Serial Monitor / Jupyter): команда `LED ON` вмикає світлодіод (`GPIO2`), `LED OFF` — вимикає.

In [ ]:
from machine import UART, Pin
import time

uart = UART(0, baudrate=115200)
led = Pin(2, Pin.OUT)

while True:
    if uart.any():
        line = uart.readline()
        if line:
            command = line.decode().strip()
            print("Отримано команду:", command)
            if command == "LED ON":
                led.on()
            elif command == "LED OFF":
                led.off()
            else:
                print("Невідома команда")
    time.sleep_ms(50)


### 🔢 Завдання:
1. Додайте команду `LED TOGGLE`, що перемикає поточний стан.
2. Додайте команду `STATUS`, у відповідь на яку плата надсилає назад поточний стан LED (`ON`/`OFF`).
3. Реалізуйте команду `BLINK <ms>`, що запускає блимання з заданим у мілісекундах періодом (використайте Timer із Завдання 2).
4. Додайте керування другим світлодіодом окремими командами `LED2 ON`/`LED2 OFF`.
5. Реалізуйте обробку команди з аргументом: `PWM <0-100>` — встановлює яскравість LED через ШІМ у відсотках.
6. Додайте перевірку на некоректний ввід (наприклад, порожній рядок) без падіння програми.
7. Реалізуйте просте «меню» — команда `HELP` виводить список усіх доступних команд.
8. Зробіть регістронезалежний розбір команд (`led on`, `LED On` тощо повинні спрацьовувати однаково).
9. Додайте журнал (список) останніх 5 отриманих команд і команду `LOG` для його виводу.
10. Поєднайте із Завданням 3: одночасно приймайте команди І надсилайте показання псевдо-сенсора.

## 📝 Завдання 5. Обмін даними між двома ESP32 по UART
З'єднайте дві плати ESP32 через `UART2` (TX↔RX навхрест) і реалізуйте простий обмін: плата А щосекунди надсилає лічильник, плата Б приймає його й виводить у свій Serial Monitor.

In [ ]:
# ---------- Код для ПЛАТИ А (передавач) ----------
from machine import UART
import time

uart = UART(2, baudrate=115200, tx=17, rx=16)
counter = 0

while True:
    counter += 1
    uart.write("COUNT:{}\n".format(counter))
    print("Надіслано:", counter)
    time.sleep(1)

# ---------- Код для ПЛАТИ Б (приймач), окремий файл ----------
# from machine import UART
# import time
#
# uart = UART(2, baudrate=115200, tx=17, rx=16)
#
# while True:
#     if uart.any():
#         line = uart.readline()
#         if line:
#             print("Отримано:", line.decode().strip())
#     time.sleep_ms(50)


### 🔢 Завдання:
1. Реалізуйте двосторонній обмін: плата Б у відповідь надсилає підтвердження `ACK:<номер>`.
2. Додайте перевірку: якщо плата Б не отримує жодного повідомлення 5 секунд поспіль, вона виводить `Немає зв'язку`.
3. Реалізуйте передачу показань кнопки з плати А (натиснута/ні) на плату Б, яка відповідно керує своїм світлодіодом.
4. Реалізуйте просту command-response взаємодію: плата Б надсилає команду `PING`, плата А відповідає `PONG`.
5. Додайте на обидві плати обробку помилок формату (рядок без очікуваного префікса `COUNT:` ігнорується).
6. Виміряйте (за допомогою `time.ticks_ms()`) затримку між надсиланням і прийомом повідомлення.
7. Розширіть протокол: передавайте одразу кілька значень через кому й розберіть їх на приймачі (`split(',')`).
8. Реалізуйте передачу даних потенціометра з плати А на плату Б, яка відображає яскравість LED відповідно (ADC на А → PWM на Б).
9. Додайте контрольну суму повідомлення і перевірку на приймачі — якщо сума не збігається, повідомлення відкидається.
10. Опишіть у markdown-комірці власний протокол обміну (формат повідомлень, коди команд) для цієї пари плат.

---

## Висновок

> В цій ...